In [ ]:
# Welcome to your new notebook

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import SparkSession, functions as F, Window
from datetime import date, timedelta


# ─────────────────────────────────────────────────────────
# CONFIG — adjust paths to match your environment
# ─────────────────────────────────────────────────────────
TICKER_SIGNALS_PATH   = "gold_ticker_signals"
SECTOR_DAILY_PATH     = "gold_sector_daily"
RANKINGS_DAILY_PATH   = "gold_rankings_daily"
INTRADAY_SUMMARY_PATH = "gold_intraday_summary"

# Set to None to auto-detect the latest date in the dataset
SELECTED_DATE = None  # e.g. date(2024, 6, 1)

# ─────────────────────────────────────────────────────────
# Load tables
# ─────────────────────────────────────────────────────────
ticker_signals   = spark.table(TICKER_SIGNALS_PATH)
sector_daily     = spark.table(SECTOR_DAILY_PATH)
rankings_daily   = spark.table(RANKINGS_DAILY_PATH)
intraday_summary = spark.table(INTRADAY_SUMMARY_PATH)

# ─────────────────────────────────────────────────────────
# Foundation date helpers  (_Latest Date, _Selected Date,
#                           _Previous Day, _Date 1W Ago, _Date 1M Ago)
# ─────────────────────────────────────────────────────────
# Replace the date helper section in Section 0 with this:

def nearest_date_before(df, ref_date):
    """Find the closest available trading date on or before ref_date."""
    row = (
        df.filter(F.col("date") <= ref_date)
          .agg(F.max("date").alias("d"))
          .collect()[0]
    )
    return row["d"]

latest_date   = sector_daily.agg(F.max("date")).collect()[0][0]
selected_date = SELECTED_DATE if SELECTED_DATE is not None else latest_date
prev_day      = nearest_date_before(sector_daily, selected_date - timedelta(days=1))
date_1w       = nearest_date_before(sector_daily, selected_date - timedelta(days=7))
date_1m       = nearest_date_before(sector_daily, selected_date - timedelta(days=30))

print(f"Selected : {selected_date}")
print(f"Prev day : {prev_day}")
print(f"1W ago   : {date_1w}")
print(f"1M ago   : {date_1m}")

# Verify all four dates have data
print("\n=== Row counts at each date ===")
for label, d in [("today", selected_date), ("prev", prev_day), ("1w", date_1w), ("1m", date_1m)]:
    count = sector_daily.filter(F.col("date") == d).count()
    print(f"  {label:6s} ({d}): {count} rows")

selected_date = SELECTED_DATE if SELECTED_DATE is not None else latest_date

# Previous day: latest date strictly before selected_date
prev_day_row = (
    ticker_signals
    .filter(F.col("date") < selected_date)
    .agg(F.max("date").alias("prev_day"))
    .collect()[0]
)

print(f"Selected date : {selected_date}")
print(f"Previous day  : {prev_day}")
print(f"1 week ago    : {date_1w}")
print(f"1 month ago   : {date_1m}")

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 5, Finished, Available, Finished, False)

Selected : 2026-06-03
Prev day : 2026-06-02
1W ago   : 2026-05-27
1M ago   : 2026-05-04

=== Row counts at each date ===
  today  (2026-06-03): 2 rows
  prev   (2026-06-02): 2 rows
  1w     (2026-05-27): 2 rows
  1m     (2026-05-04): 2 rows
Selected date : 2026-06-03
Previous day  : 2026-06-02
1 week ago    : 2026-05-27
1 month ago   : 2026-05-04


In [4]:
SIGNAL_COLS = [
    "signal_rsi_overbought",
    "signal_rsi_oversold",
    "signal_macd_bullish_cross",
    "signal_macd_bearish_cross",
    "signal_golden_cross",
    "signal_death_cross",
    "signal_vol_spike",
    "signal_bb_upper_touch",
    "signal_bb_lower_touch",
]

def count_signals_on_date(df, on_date):
    """Count each signal col + any_signal, grouped by sector, for a given date."""
    agg_exprs = [
        F.sum(F.col(c).cast("int")).alias(c)
        for c in SIGNAL_COLS
    ]
    agg_exprs.append(
        F.sum(F.col("any_signal").cast("int")).alias("tickers_with_signal")
    )
    return (
        df.filter(F.col("date") == on_date)
          .groupBy("sector")
          .agg(*agg_exprs)
    )

signals_today = count_signals_on_date(ticker_signals, selected_date)
signals_prev  = count_signals_on_date(ticker_signals, prev_day)

# Suffix prev columns then join
signals_prev_renamed = signals_prev.select(
    "sector",
    F.col("tickers_with_signal").alias("tickers_with_signal_prev")
)

signal_summary = (
    signals_today
    .join(signals_prev_renamed, "sector", "left")
    .withColumn(
        "Signal Count Change vs Prev Day",
        F.col("tickers_with_signal") - F.col("tickers_with_signal_prev")
    )
)

print("=== Signal Summary ===")
signal_summary.show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 6, Finished, Available, Finished, False)

=== Signal Summary ===
+-----------+---------------------+-------------------+-------------------------+-------------------------+-------------------+------------------+----------------+---------------------+---------------------+-------------------+------------------------+-------------------------------+
|sector     |signal_rsi_overbought|signal_rsi_oversold|signal_macd_bullish_cross|signal_macd_bearish_cross|signal_golden_cross|signal_death_cross|signal_vol_spike|signal_bb_upper_touch|signal_bb_lower_touch|tickers_with_signal|tickers_with_signal_prev|Signal Count Change vs Prev Day|
+-----------+---------------------+-------------------+-------------------------+-------------------------+-------------------+------------------+----------------+---------------------+---------------------+-------------------+------------------------+-------------------------------+
|Health Care|0                    |1                  |1                        |0                        |0              

In [5]:
# Helper: average a column for a specific date, grouped by sector
def sector_avg_on_date(df, col_name, on_date, alias_suffix):
    return (
        df.filter(F.col("date") == on_date)
          .groupBy("sector")
          .agg(F.avg(col_name).alias(f"{col_name}_{alias_suffix}"))
    )

# ── RSI ──────────────────────────────────────────────────
rsi_today  = sector_avg_on_date(sector_daily, "avg_rsi_14", selected_date, "today")
rsi_prev   = sector_avg_on_date(sector_daily, "avg_rsi_14", prev_day,       "prev")
rsi_1w     = sector_avg_on_date(sector_daily, "avg_rsi_14", date_1w,        "1w")
rsi_1m     = sector_avg_on_date(sector_daily, "avg_rsi_14", date_1m,        "1m")

rsi_health = (
    rsi_today
    .join(rsi_prev, "sector", "left")
    .join(rsi_1w,   "sector", "left")
    .join(rsi_1m,   "sector", "left")
    .withColumn("RSI Change vs Prev Day", F.col("avg_rsi_14_today") - F.col("avg_rsi_14_prev"))
    .withColumn("RSI Change vs 1W",       F.col("avg_rsi_14_today") - F.col("avg_rsi_14_1w"))
    .withColumn("RSI Change vs 1M",       F.col("avg_rsi_14_today") - F.col("avg_rsi_14_1m"))
    .select(
        "sector",
        F.col("avg_rsi_14_today").alias("Avg RSI"),
        F.col("avg_rsi_14_prev").alias("Avg RSI Prev Day"),
        F.col("avg_rsi_14_1w").alias("Avg RSI 1W Ago"),
        F.col("avg_rsi_14_1m").alias("Avg RSI 1M Ago"),
        "RSI Change vs Prev Day",
        "RSI Change vs 1W",
        "RSI Change vs 1M",
    )
)

# ── Daily Return ─────────────────────────────────────────
ret_today  = sector_avg_on_date(sector_daily, "avg_daily_return", selected_date, "today")
ret_prev   = sector_avg_on_date(sector_daily, "avg_daily_return", prev_day,       "prev")
ret_1w     = sector_avg_on_date(sector_daily, "avg_daily_return", date_1w,        "1w")
ret_1m     = sector_avg_on_date(sector_daily, "avg_daily_return", date_1m,        "1m")

return_health = (
    ret_today
    .join(ret_prev, "sector", "left")
    .join(ret_1w,   "sector", "left")
    .join(ret_1m,   "sector", "left")
    .withColumn("Return Change vs Prev Day", F.col("avg_daily_return_today") - F.col("avg_daily_return_prev"))
    .withColumn("Return Change vs 1W",       F.col("avg_daily_return_today") - F.col("avg_daily_return_1w"))
    .withColumn("Return Change vs 1M",       F.col("avg_daily_return_today") - F.col("avg_daily_return_1m"))
    .select(
        "sector",
        F.col("avg_daily_return_today").alias("Avg Daily Return"),
        F.col("avg_daily_return_prev").alias("Avg Daily Return Prev Day"),
        F.col("avg_daily_return_1w").alias("Avg Daily Return 1W Ago"),
        F.col("avg_daily_return_1m").alias("Avg Daily Return 1M Ago"),
        "Return Change vs Prev Day",
        "Return Change vs 1W",
        "Return Change vs 1M",
    )
)

# ── Advance / Decline Ratio ───────────────────────────────
adr_today = sector_avg_on_date(sector_daily, "advance_decline_ratio", selected_date, "today")
adr_prev  = sector_avg_on_date(sector_daily, "advance_decline_ratio", prev_day,       "prev")

adr_health = (
    adr_today
    .join(adr_prev, "sector", "left")
    .withColumn("AD Ratio Change vs Prev Day",
                F.col("advance_decline_ratio_today") - F.col("advance_decline_ratio_prev"))
    .select(
        "sector",
        F.col("advance_decline_ratio_today").alias("Advance Decline Ratio"),
        F.col("advance_decline_ratio_prev").alias("AD Ratio Prev Day"),
        "AD Ratio Change vs Prev Day",
    )
)

# ── Advancing / Declining / Unchanged Ticker Counts ──────
ticker_counts = (
    sector_daily
    .filter(F.col("date") == selected_date)
    .groupBy("sector")
    .agg(
        F.sum("advancing_tickers").alias("Advancing Tickers"),
        F.sum("declining_tickers").alias("Declining Tickers"),
        F.sum("unchanged_tickers").alias("Unchanged Tickers"),
    )
)

print("=== RSI Health ===")
rsi_health.show(truncate=False)

print("=== Return Health ===")
return_health.show(truncate=False)

print("=== Advance/Decline ===")
adr_health.show(truncate=False)

print("=== Ticker Counts ===")
ticker_counts.show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 7, Finished, Available, Finished, False)

=== RSI Health ===
+-----------+-------+----------------+--------------+--------------+----------------------+------------------+------------------+
|sector     |Avg RSI|Avg RSI Prev Day|Avg RSI 1W Ago|Avg RSI 1M Ago|RSI Change vs Prev Day|RSI Change vs 1W  |RSI Change vs 1M  |
+-----------+-------+----------------+--------------+--------------+----------------------+------------------+------------------+
|Health Care|45.27  |41.91           |50.51         |33.98         |3.3600000000000065    |-5.239999999999995|11.290000000000006|
|Industrials|47.11  |47.77           |49.35         |44.53         |-0.6600000000000037   |-2.240000000000002|2.5799999999999983|
+-----------+-------+----------------+--------------+--------------+----------------------+------------------+------------------+

=== Return Health ===
+-----------+----------------+-------------------------+-----------------------+-----------------------+-------------------------+-------------------+--------------------+
|secto

In [6]:
rankings_today = rankings_daily.filter(F.col("date") == selected_date)

# Top Ticker Today (rank 1 within sector)
top_tickers = (
    rankings_today
    .filter(F.col("return_rank_in_sector") == 1)
    .groupBy("sector")
    .agg(F.first("ticker").alias("Top Ticker Today"))
)

# Best / Worst return and Avg Composite Rank
perf_summary = (
    rankings_today
    .groupBy("sector")
    .agg(
        F.max("daily_return").alias("Best Return Today"),
        F.min("daily_return").alias("Worst Return Today"),
        F.avg("composite_rank_score").alias("Avg Composite Rank"),
    )
)

# Return Spread (from sector_daily)
spread_today = (
    sector_daily
    .filter(F.col("date") == selected_date)
    .groupBy("sector")
    .agg(F.avg("return_spread").alias("Return Spread Today"))
)
spread_1w = (
    sector_daily
    .filter(F.col("date") == date_1w)
    .groupBy("sector")
    .agg(F.avg("return_spread").alias("Return Spread 1W Ago"))
)

spread_comparison = (
    spread_today
    .join(spread_1w, "sector", "left")
    .withColumn(
        "Return Spread Change vs 1W",
        F.col("Return Spread Today") - F.col("Return Spread 1W Ago")
    )
)

performance = (
    perf_summary
    .join(top_tickers,       "sector", "left")
    .join(spread_comparison, "sector", "left")
)

print("=== Performance / Rankings ===")
performance.show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 8, Finished, Available, Finished, False)

=== Performance / Rankings ===
+-----------+-----------------+------------------+------------------+----------------+-------------------+--------------------+--------------------------+
|sector     |Best Return Today|Worst Return Today|Avg Composite Rank|Top Ticker Today|Return Spread Today|Return Spread 1W Ago|Return Spread Change vs 1W|
+-----------+-----------------+------------------+------------------+----------------+-------------------+--------------------+--------------------------+
|Health Care|0.074934         |-0.008219         |8.0               |MRNA            |0.083153           |0.141641            |-0.058487999999999984     |
|Industrials|0.012436         |-0.029947         |7.0               |CHRW            |0.042383           |0.09542             |-0.05303700000000001      |
+-----------+-----------------+------------------+------------------+----------------+-------------------+--------------------+--------------------------+



In [7]:
vol_today = sector_avg_on_date(sector_daily, "avg_volatility", selected_date, "today")
vol_prev  = sector_avg_on_date(sector_daily, "avg_volatility", prev_day,       "prev")
vol_1w    = sector_avg_on_date(sector_daily, "avg_volatility", date_1w,        "1w")
vol_1m    = sector_avg_on_date(sector_daily, "avg_volatility", date_1m,        "1m")

volatility = (
    vol_today
    .join(vol_prev, "sector", "left")
    .join(vol_1w,   "sector", "left")
    .join(vol_1m,   "sector", "left")
    .withColumn("Volatility Change vs Prev Day",
                F.col("avg_volatility_today") - F.col("avg_volatility_prev"))
    .withColumn("Volatility Change vs 1W",
                F.col("avg_volatility_today") - F.col("avg_volatility_1w"))
    .withColumn("Volatility Change vs 1M",
                F.col("avg_volatility_today") - F.col("avg_volatility_1m"))
    # Volatility Regime: compare today vs 1M baseline
    .withColumn(
        "Volatility Regime",
        F.when(F.col("avg_volatility_today").isNull(), None)
         .when(F.col("avg_volatility_today") > F.col("avg_volatility_1m") * 1.5,  "🔴 High")
         .when(F.col("avg_volatility_today") < F.col("avg_volatility_1m") * 0.75, "🟢 Low")
         .otherwise("🟡 Normal")
    )
    .select(
        "sector",
        F.col("avg_volatility_today").alias("Avg Volatility"),
        F.col("avg_volatility_prev").alias("Avg Volatility Prev Day"),
        F.col("avg_volatility_1w").alias("Avg Volatility 1W Ago"),
        F.col("avg_volatility_1m").alias("Avg Volatility 1M Ago"),
        "Volatility Change vs Prev Day",
        "Volatility Change vs 1W",
        "Volatility Change vs 1M",
        "Volatility Regime",
    )
)

print("=== Volatility ===")
volatility.show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 9, Finished, Available, Finished, False)

=== Volatility ===
+-----------+--------------+-----------------------+---------------------+---------------------+-----------------------------+-----------------------+-----------------------+-----------------+
|sector     |Avg Volatility|Avg Volatility Prev Day|Avg Volatility 1W Ago|Avg Volatility 1M Ago|Volatility Change vs Prev Day|Volatility Change vs 1W|Volatility Change vs 1M|Volatility Regime|
+-----------+--------------+-----------------------+---------------------+---------------------+-----------------------------+-----------------------+-----------------------+-----------------+
|Health Care|0.020082      |0.019379               |0.020826             |0.019319             |7.029999999999988E-4         |-7.440000000000016E-4  |7.629999999999998E-4   |🟡 Normal        |
|Industrials|0.024576      |0.027767               |0.030432             |0.024226             |-0.0031909999999999994       |-0.005856              |3.499999999999996E-4   |🟡 Normal        |
+-----------+-----

In [8]:
intraday_today = intraday_summary.filter(F.col("date") == selected_date)

def session_agg(df, session_name):
    """Avg return and total volume for a named session, by sector."""
    return (
        df.filter(F.col("session") == session_name)
          .groupBy("sector")
          .agg(
              F.avg("session_return").alias(f"{session_name}_return"),
              F.sum("session_volume").alias(f"{session_name}_volume"),
          )
    )

pre_agg     = session_agg(intraday_today, "pre")
regular_agg = session_agg(intraday_today, "regular")
after_agg   = session_agg(intraday_today, "after")

intraday = (
    pre_agg
    .join(regular_agg, "sector", "left")
    .join(after_agg,   "sector", "left")
    # Pre vs Regular Gap: did pre-market momentum carry through?
    # Positive = same direction; Negative = reversal during regular hours
    .withColumn(
        "Pre vs Regular Gap",
        F.col("pre_return") - F.col("regular_return")
    )
    .select(
        "sector",
        F.col("pre_return").alias("Pre Session Return"),
        F.col("regular_return").alias("Regular Session Return"),
        F.col("after_return").alias("After Session Return"),
        "Pre vs Regular Gap",
        F.col("pre_volume").alias("Pre Session Volume"),
        F.col("regular_volume").alias("Regular Session Volume"),
    )
)

print("=== Intraday Sessions ===")
intraday.show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 10, Finished, Available, Finished, False)

=== Intraday Sessions ===
+-----------+---------------------+----------------------+---------------------+---------------------+------------------+----------------------+
|sector     |Pre Session Return   |Regular Session Return|After Session Return |Pre vs Regular Gap   |Pre Session Volume|Regular Session Volume|
+-----------+---------------------+----------------------+---------------------+---------------------+------------------+----------------------+
|Health Care|0.0020496666666666667|0.015265333333333332  |0.0036771333333333327|-0.013215666666666665|959853.6282610003 |9.706376251559E7      |
|Industrials|-0.001585            |-0.004011153846153847 |-9.232500000000001E-4|0.0024261538461538465|29614.927249      |1.2090726281011999E7  |
+-----------+---------------------+----------------------+---------------------+---------------------+------------------+----------------------+



In [9]:
# Pivot the sector health data to get Shipping and Pharma side-by-side
SECTOR_A = "Health Care"
SECTOR_B = "Industrials"
sector_pivot = (
    sector_daily
    .filter(
        (F.col("date") == selected_date) &
        (F.col("sector").isin(SECTOR_A, SECTOR_B))
    )
    .groupBy("sector")
    .agg(
        F.avg("avg_rsi_14").alias("avg_rsi"),
        F.avg("avg_volatility").alias("avg_vol"),
        F.avg("avg_daily_return").alias("avg_return"),
    )
)

# Collect into Python for the scalar gap calculations
sector_vals = {row["sector"]: row for row in sector_pivot.collect()}

shipping = sector_vals.get(SECTOR_B)
pharma   = sector_vals.get(SECTOR_A)

def gap(field):
    s = shipping[field] if shipping else None
    p = pharma[field]   if pharma   else None
    return round(s - p, 4) if (s is not None and p is not None) else None

rsi_gap        = gap("avg_rsi")
volatility_gap = gap("avg_vol")
return_gap     = gap("avg_return")

# Sector Alignment label
ship_ret = shipping["avg_return"] if shipping else None
phar_ret = pharma["avg_return"]   if pharma   else None

if ship_ret is None or phar_ret is None:
    alignment = "N/A"
elif ship_ret > 0 and phar_ret > 0:
    alignment = "📈 Both Rising"
elif ship_ret < 0 and phar_ret < 0:
    alignment = "📉 Both Falling"
elif ship_ret > 0 and phar_ret < 0:
    alignment = "🔀 Shipping Up / Pharma Down"
else:
    alignment = "🔀 Pharma Up / Shipping Down"

print("=== Cross-Sector Comparison ===")
print(f"  RSI Gap        (Shipping - Pharma) : {rsi_gap}")
print(f"  Volatility Gap (Shipping - Pharma) : {volatility_gap}")
print(f"  Return Gap     (Shipping - Pharma) : {return_gap}")
print(f"  Sector Alignment                   : {alignment}")

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 11, Finished, Available, Finished, False)

=== Cross-Sector Comparison ===
  RSI Gap        (Shipping - Pharma) : 1.84
  Volatility Gap (Shipping - Pharma) : 0.0045
  Return Gap     (Shipping - Pharma) : -0.024
  Sector Alignment                   : 🔀 Pharma Up / Shipping Down


In [1]:
# Join every measure table on 'sector'
consolidated = (
    rsi_health
    .join(return_health,  "sector", "left")
    .join(adr_health,     "sector", "left")
    .join(ticker_counts,  "sector", "left")
    .join(signal_summary, "sector", "left")
    .join(performance,    "sector", "left")
    .join(volatility,     "sector", "left")
    .join(intraday,       "sector", "left")
)

import re

def clean_col_names(df):
    for col in df.columns:
        clean = re.sub(r'[ ,;{}()\n\t=]', '_', col)
        if clean != col:
            df = df.withColumnRenamed(col, clean)
    return df

consolidated_clean = clean_col_names(consolidated)

consolidated_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dashboard_measures")

print("✅ Saved to dashboard_snapshot")
print(f"   Rows: {consolidated_clean.count()}")
print(f"   Cols: {len(consolidated_clean.columns)}")
consolidated_clean.show(truncate=False)


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [11]:
print(f"selected_date : {selected_date}")
print(f"prev_day      : {prev_day}")
print(f"date_1w       : {date_1w}")
print(f"date_1m       : {date_1m}")
print()

# Check how many rows each source table has on selected_date
print("=== Row counts on selected_date ===")
print("ticker_signals  :", ticker_signals.filter(F.col("date") == selected_date).count())
print("sector_daily    :", sector_daily.filter(F.col("date") == selected_date).count())
print("rankings_daily  :", rankings_daily.filter(F.col("date") == selected_date).count())
print("intraday_summary:", intraday_summary.filter(F.col("date") == selected_date).count())
print()

# Check what sectors actually exist in sector_daily
print("=== Sectors in sector_daily ===")
sector_daily.select("sector").distinct().show(truncate=False)

# Check what the date column actually looks like
print("=== Sample dates in sector_daily ===")
sector_daily.select("date").distinct().orderBy("date", ascending=False).show(10, truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 13, Finished, Available, Finished, False)

selected_date : 2026-06-03
prev_day      : 2026-06-02
date_1w       : 2026-05-27
date_1m       : 2026-05-04

=== Row counts on selected_date ===
ticker_signals  : 28
sector_daily    : 2
rankings_daily  : 28
intraday_summary: 77

=== Sectors in sector_daily ===
+-----------+
|sector     |
+-----------+
|Health Care|
|Industrials|
+-----------+

=== Sample dates in sector_daily ===
+----------+
|date      |
+----------+
|2026-06-03|
|2026-06-02|
|2026-06-01|
|2026-05-29|
|2026-05-28|
|2026-05-27|
|2026-05-26|
|2026-05-22|
|2026-05-21|
|2026-05-20|
+----------+
only showing top 10 rows



In [ ]:
import requests
import sempy.fabric as fabric


WORKSPACE_ID = fabric.get_workspace_id()


token = mssparkutils.credentials.getToken("pbi")

# List all datasets in the workspace
response = requests.get(
    f"https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets",
    headers={"Authorization": f"Bearer {token}"}
)

datasets = response.json()["value"]
DATASET_ID   = datasets[0]["id"]

response = requests.post(
    f"https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets/{DATASET_ID}/refreshes",
    headers={"Authorization": f"Bearer {token}"},
    json={"notifyOption": "NoNotification"}
)

print("Refresh triggered" if response.status_code == 202 else f"Failed: {response.status_code} — {response.text}")

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 14, Finished, Available, Finished, False)

Refresh triggered


In [13]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 15, Finished, Available, Finished, False)

+-----------+----------------------+-----------+
|namespace  |tableName             |isTemporary|
+-----------+----------------------+-----------+
|Avi.Avi.dbo|bronze_bars           |false      |
|Avi.Avi.dbo|bronze_bars_v2        |false      |
|Avi.Avi.dbo|dashboard_measures    |false      |
|Avi.Avi.dbo|dashboard_snapshot    |false      |
|Avi.Avi.dbo|gold_correlation_daily|false      |
|Avi.Avi.dbo|gold_intraday_summary |false      |
|Avi.Avi.dbo|gold_rankings_daily   |false      |
|Avi.Avi.dbo|gold_sector_daily     |false      |
|Avi.Avi.dbo|gold_ticker_signals   |false      |
|Avi.Avi.dbo|silver_1min           |false      |
|Avi.Avi.dbo|silver_5min           |false      |
|Avi.Avi.dbo|silver_daily          |false      |
+-----------+----------------------+-----------+



In [14]:
print("=== Column types ===")
sector_daily.printSchema()
ticker_signals.printSchema()

# Check what the actual values look like
sector_daily.select("date").orderBy("date", ascending=False).show(5, truncate=False)
ticker_signals.select("date").orderBy("date", ascending=False).show(5, truncate=False)

# Check if date_1m exists in the data
print(f"date_1m = {date_1m}, type = {type(date_1m)}")
sector_daily.filter(F.col("date") == date_1m).show(5)

StatementMeta(, 038e77ad-23ed-4622-9cf9-ab19813f6255, 16, Finished, Available, Finished, False)

=== Column types ===
root
 |-- sector: string (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- is_month_end: boolean (nullable = true)
 |-- is_quarter_end: boolean (nullable = true)
 |-- ticker_count: long (nullable = true)
 |-- avg_open: double (nullable = true)
 |-- avg_close: double (nullable = true)
 |-- sector_high: double (nullable = true)
 |-- sector_low: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- advancing_tickers: long (nullable = true)
 |-- declining_tickers: long (nullable = true)
 |-- unchanged_tickers: long (nullable = true)
 |-- avg_daily_return: double (nullable = true)
 |-- avg_rsi_14: double (nullable = true)
 |-- avg_macd: double (nullable = true)
 |-- avg_volatility: double (nullable